# Orientation Bug Replication

Creates synthetic ellipse polygons at **known** angles and runs them through both orientation paths:

1. **`fitEllipse`** (scikit-image `EllipseModel`) → returns `theta` in radians
2. **`boulder_row`** (Shapely minimum rotated rectangle → `azimuth()`) → returns `angle`/`angle180`

For each input angle we compare what the code returns vs. what it should return.
If a column is constant regardless of input → that path contains the bug.

In [ ]:
import numpy as np
import pandas as pd
from shapely.geometry import Polygon

from shptools_BOULDERING.geometry import fitEllipse
from shptools_BOULDERING.geomorph import boulder_row, azimuth

## Helper: generate a synthetic ellipse polygon at a known angle

In [ ]:
def make_ellipse_polygon(a, b, theta_degrees, cx=0.0, cy=0.0, n_pts=200):
    """
    Return a Shapely Polygon of an ellipse with:
      a            : semi-major axis length
      b            : semi-minor axis length  (b < a)
      theta_degrees: rotation CCW from positive x-axis (East)
      cx, cy       : center coordinates
    """
    theta_rad = np.radians(theta_degrees)
    t = np.linspace(0, 2 * np.pi, n_pts, endpoint=False)
    x = a * np.cos(t)
    y = b * np.sin(t)
    # rotate CCW by theta
    x_rot = x * np.cos(theta_rad) - y * np.sin(theta_rad) + cx
    y_rot = x * np.sin(theta_rad) + y * np.cos(theta_rad) + cy
    return Polygon(np.column_stack([x_rot, y_rot]))

## Test parameters

Use a/b = 1.5 (within the 1.2–2 range of interest).
Center is at a large coordinate to mimic real projected coordinates.

In [ ]:
A   = 15.0      # semi-major axis (pixels / map units)
B   = 10.0      # semi-minor axis  →  a/b = 1.5
CX  = 500.0     # center x (change to realistic projected coords on Sherlock if needed)
CY  = 500.0     # center y

test_angles = [0, 15, 30, 45, 60, 75, 90, 105, 120, 135, 150, 165]

print(f"Ellipse: a={A}, b={B}, a/b={A/B:.2f}, center=({CX}, {CY})")

## Run both orientation paths for every test angle

In [ ]:
results = []

for deg in test_angles:
    poly = make_ellipse_polygon(A, B, deg, cx=CX, cy=CY)
    row  = pd.Series({"geometry": poly})

    # --- Path 1: fitEllipse (EllipseModel) ---
    try:
        _, a_fit, b_fit, theta_rad = fitEllipse(row)
        theta_deg = np.degrees(theta_rad)
        fit_ok = True
    except Exception as e:
        theta_deg, a_fit, b_fit = None, None, None
        fit_ok = False
        print(f"fitEllipse failed at {deg}°: {e}")

    # --- Path 2: boulder_row (MRR + azimuth) ---
    try:
        mrr_poly = poly.minimum_rotated_rectangle
        mrr_row  = pd.Series({"geometry": mrr_poly})
        (length, width, long_axis, short_axis, diameter,
         angle_default, angle360, angle180) = boulder_row(mrr_row)
        mrr_ok = True
    except Exception as e:
        angle_default, angle360, angle180 = None, None, None
        mrr_ok = False
        print(f"boulder_row failed at {deg}°: {e}")

    results.append({
        "input_deg"      : deg,
        "fitEllipse_theta_deg": theta_deg,
        "fitEllipse_a"   : a_fit,
        "fitEllipse_b"   : b_fit,
        "mrr_angle_default": angle_default,
        "mrr_angle180"   : angle180,
    })

df = pd.DataFrame(results)
df

## What to look for

- **`fitEllipse_theta_deg`** should vary and match `input_deg` (mod 180) — if it's constant, `EllipseModel` is the problem.
- **`mrr_angle180`** should vary and match `input_deg` (mod 180) — if it's constant, `boulder_row`/`azimuth` is the problem.
- **`fitEllipse_a`** should be ≈ 15.0 and `fitEllipse_b` ≈ 10.0 — if swapped or wrong, check axis ordering in EllipseModel.

Note: both angles are expected mod 180 because an undirected axis at θ° is indistinguishable from θ+180°.

In [ ]:
# Quick visual check: plot expected vs returned
import matplotlib.pyplot as plt

expected = [d % 180 for d in test_angles]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(test_angles, expected, 'k--', label='expected (input mod 180)')
axes[0].plot(test_angles, df["fitEllipse_theta_deg"] % 180, 'ro-', label='fitEllipse theta')
axes[0].set_xlabel("Input angle (°)")
axes[0].set_ylabel("Returned angle (°)")
axes[0].set_title("Path 1: fitEllipse (EllipseModel)")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(test_angles, expected, 'k--', label='expected (input mod 180)')
axes[1].plot(test_angles, df["mrr_angle180"], 'bs-', label='MRR angle180')
axes[1].set_xlabel("Input angle (°)")
axes[1].set_ylabel("Returned angle (°)")
axes[1].set_title("Path 2: boulder_row (MRR + azimuth)")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig("orientation_bug_replication.png", dpi=150)
plt.show()
print("Plot saved to orientation_bug_replication.png")